# Task2: Data Cleaning

Step1. Data Quality Report

Load Dataset

In [4]:
import pandas as pd
import numpy as np 

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160) 

df= pd.read_csv("crime_incidents_messy.csv")
df.head(10)
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5250 entries, 0 to 5249
Data columns (total 33 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   incident_id         5250 non-null   object 
 1   crime_type          5250 non-null   object 
 2   district            5250 non-null   object 
 3   city                5250 non-null   object 
 4   state               5250 non-null   object 
 5   address             5250 non-null   object 
 6   latitude            4992 non-null   float64
 7   longitude           4961 non-null   float64
 8   incident_datetime   4910 non-null   object 
 9   officer_id          5250 non-null   object 
 10  officer_first_name  5250 non-null   object 
 11  officer_last_name   5250 non-null   object 
 12  badge_number        4938 non-null   float64
 13  suspect_id          4440 non-null   object 
 14  suspect_first_name  4440 non-null   object 
 15  suspect_last_name   4440 non-null   object 
 16  suspec

Null Values Count

In [5]:
quality_report = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum()/len(df)*100).round(1),
    'n_unique': df.nunique()
})
quality_report 

,dtype,null_count,null_pct,n_unique
incident_id,object,0,0.0,5050
crime_type,object,0,0.0,182
district,object,0,0.0,131
city,object,0,0.0,8
state,object,0,0.0,10
address,object,0,0.0,4895
latitude,float64,258,4.9,4750
longitude,float64,289,5.5,4726
incident_datetime,object,340,6.5,4675
officer_id,object,0,0.0,150


##### Observation:
###### 29/33 columns contains null-the highest proportion is in suspect_race (30.5%).crime_type contains 182 unique values but the real categories are only 18 mean there are many spelling/case variants

Check Duplicate Rows

In [6]:
duplicate_rows = df.duplicated().sum()
id_duplicates = df["incident_id"].duplicated().sum()
print(f"Number of duplicate incident IDs: {id_duplicates}")
print(f"Number of duplicate rows: {duplicate_rows}")


Number of duplicate incident IDs: 200
Number of duplicate rows: 200


Data Type Issues

In [7]:
expected_dtype = {
    'incident_id':'string','crime_type':'string','district':'string','city':'string','state':'string',
    'address':'string','latitude':'float','longitude':'float','incident_datetime':'datetime',
    'officer_id':'string','officer_first_name':'string','officer_last_name':'string','badge_number':'integer',
    'suspect_id':'string','suspect_first_name':'string','suspect_last_name':'string','suspect_age':'integer',
    'suspect_gender':'string','suspect_race':'string','victim_id':'string','victim_first_name':'string',
    'victim_last_name':'string','victim_age':'integer','victim_gender':'string','victim_phone':'string',
    'weapon_used':'string','severity':'string','case_status':'string','resolution':'string',
    'num_arrests':'integer','property_loss_usd':'float','reported_online':'boolean','notes':'string'
}
dtype_issues = pd.DataFrame({
    'Column': list(expected_dtype.keys()),
    'Current dtype': [str(df[c].dtype) for c in expected_dtype],
    'Should be': list(expected_dtype.values()),
})
dtype_issues['Issue?'] = (
    ((dtype_issues['Should be']=='datetime') & (~dtype_issues['Current dtype'].str.contains('datetime'))) |
    ((dtype_issues['Should be']=='float')   & (dtype_issues['Current dtype']!='float64')) |
    ((dtype_issues['Should be']=='integer') & (~dtype_issues['Current dtype'].isin(['int64','Int64']))) |
    ((dtype_issues['Should be']=='boolean') & (dtype_issues['Current dtype']!='bool'))
)

dtype_issues_found = dtype_issues[dtype_issues['Issue?']]
dtype_issues_found

,Column,Current dtype,Should be,Issue?
8,incident_datetime,object,datetime,True
12,badge_number,float64,integer,True
16,suspect_age,float64,integer,True
22,victim_age,float64,integer,True
29,num_arrests,float64,integer,True
30,property_loss_usd,object,float,True
31,reported_online,object,boolean,True


##### Observation:

###### The dtype of 7 columns are wrong: incident_datetime (is text , we wantdatetime),badge_number/suspect_age/victim_age/num_arrests (is float, we want integer), property_loss_usd (is text, we want float), reported_online (is text, we want boolean).

Value Range Anomalies

In [8]:
range_anomalies= pd.DataFrame({
    'Column': ['latitude','longitude','suspect_age','victim_age','num_arrests','property_loss_usd'],
    'Expected range': ['-90 to 90','-180 to 180','0 to 100','0 to 100','0 or more','0 or more'],
    'Actual min': [df['latitude'].min(), df['longitude'].min(), df['suspect_age'].min(),
                   df['victim_age'].min(), df['num_arrests'].min(),
                   pd.to_numeric(df['property_loss_usd'], errors='coerce').min()],
    
    'Actual max': [df['latitude'].max(), df['longitude'].max(), df['suspect_age'].max(),
                   df['victim_age'].max(), df['num_arrests'].max(),
                   pd.to_numeric(df['property_loss_usd'], errors='coerce').max()], 
})
range_anomalies 

,Column,Expected range,Actual min,Actual max
0,latitude,-90 to 90,25.002420,199.9842
1,longitude,-180 to 180,-121.975849,99.7873
2,suspect_age,0 to 100,-75.000000,298.0000
3,victim_age,0 to 100,-90.000000,298.0000
4,num_arrests,0 or more,-5.000000,5.0000
5,property_loss_usd,0 or more,-49866.740000,49998.3000


##### Observation

###### The latitude goes to 199.98(but 90 is max valid, impossible), suspect_age/victim_age are of -75 se 298(not an age of human), num_arrests & property_loss_usd are also negative(both things can never be negative)

Step2: Duplicate Removal

In [9]:
rows_before = len(df)

dup_count = df.duplicated(subset=['incident_id']).sum()
df = df.drop_duplicates(subset=['incident_id'], keep='first').reset_index(drop=True)

rows_after = len(df)

dedup_summary = pd.DataFrame({
    'Metric': ['Rows before', 'Duplicates removed', 'Rows after'],
    'Value': [rows_before, dup_count, rows_after]
})
dedup_summary                 

,Metric,Value
0,Rows before,5250
1,Duplicates removed,200
2,Rows after,5050


Step3: Standardization

In [10]:
df_original = pd.read_csv('crime_incidents_messy.csv')  

text_cols = ['crime_type', 'district', 'suspect_gender', 'victim_gender',
             'suspect_race', 'case_status', 'resolution', 'severity']

for c in text_cols:
    print(f"\n{'='*50}")
    print(f"Column: {c}")
    print(f"Number of unique values: {df_original[c].nunique()}")
    print(f"{'='*50}")
    
    vc_table = df_original[c].value_counts(dropna=False).reset_index()
    vc_table.columns = ['Value', 'Count']
    display(vc_table)


Column: crime_type
Number of unique values: 182


,Value,Count
0,Fire Setting,84
1,DV,82
2,ARSON,80
3,Drug Offence,79
4,Deception,79
...,...,...
177,Sex Assault,1
178,assault & battery,1
179,CYBER CRIME,1
180,dui,1



Column: district
Number of unique values: 131


,Value,Count
0,Nor,273
1,Sou,265
2,SOUTHWEST,106
3,South,102
4,NORTH,101
...,...,...
126,Southeast,1
127,West,1
128,southeast,1
129,mid,1



Column: suspect_gender
Number of unique values: 12


...

,Value,Count
0,NaN,731
1,closed,399
2,Resolved,389
3,Under Investigation,387
4,Open,381
5,open,380
6,Investgation,377
7,Closed,374
8,OPEN,371
9,Pendng,370



Column: resolution
Number of unique values: 9


,Value,Count
0,NaN,951
1,Arrest Made,504
2,arrest made,491
3,No Arrest,490
4,warning,490
5,Dismissed,480
6,Warning Issued,479
7,Arres Made,476
8,Case Dismissed,455
9,NO ARREST,434



Column: severity
Number of unique values: 14


,Value,Count
0,4,376
1,MEDIUM,368
2,high,366
3,low,366
4,1,363
5,NaN,355
6,Med,355
7,3,348
8,Low,347
9,Medium,347


In [11]:
text_cols = ['crime_type','district','city','state','address','officer_first_name','officer_last_name',
             'suspect_first_name','suspect_last_name','victim_first_name','victim_last_name',
             'suspect_gender','victim_gender','suspect_race','case_status','resolution','weapon_used',
             'severity','notes']


for c in text_cols:
    df[c] = df[c].str.strip().str.replace(r'\s+', ' ', regex=True)
   

crime_map = {'roberry':'Robbery','robbry':'Robbery','robbery':'Robbery','armed robbery':'Robbery',
    'burglry':'Burglary','burglary':'Burglary','homocide':'Homicide','homicide':'Homicide',
    'murder':'Homicide','manslaughter':'Homicide','kidnaping':'Kidnapping','kidnapping':'Kidnapping',
    'abduction':'Kidnapping','vandlism':'Vandalism','vandalism':'Vandalism','arsen':'Arson',
    'arson':'Arson','fire setting':'Arson','dv':'Domestic Violence','dom. violence':'Domestic Violence',
    'domestc violence':'Domestic Violence','domestic violence':'Domestic Violence',
    'tresspassing':'Trespassing','trespassing':'Trespassing','trespass':'Trespassing',
    'sexual assualt':'Sexual Assault','sexual assault':'Sexual Assault','sex assault':'Sexual Assault',
    'sa':'Sexual Assault','b&e':'Breaking & Entering','breaking & entering':'Breaking & Entering',
    'dui':'DUI','d.u.i.':'DUI','duii':'DUI','dwi':'DUI','drunk driving':'DUI',
    'drug offence':'Drug Offense','drug offense':'Drug Offense','drugs':'Drug Offense','narcotics':'Drug Offense',
    'theft/larceny':'Larceny/Theft','theft':'Larceny/Theft','larceny':'Larceny/Theft','stealing':'Larceny/Theft',
    'cyber crime':'Cybercrime','cybercrime':'Cybercrime','hacking':'Cybercrime','online fraud':'Fraud',
    'fraudulent activity':'Fraud','scam':'Fraud','deception':'Fraud','fraud':'Fraud',
    'assault & battery':'Assault','battery':'Assault','assault':'Assault','asslt':'Assault',
    'graffiti':'Graffiti','property damage':'Property Damage'}
df['crime_type'] = df['crime_type'].str.lower().map(crime_map).fillna(df['crime_type'].str.title())


district_map = {'nor':'North','sou':'South','cen':'Central','eas':'East','wes':'West','mid':'Midtown',
    'north':'North','south':'South','east':'East','west':'West','central':'Central','midtown':'Midtown',
    'northeast':'Northeast','northwest':'Northwest','southeast':'Southeast','southwest':'Southwest'}
df['district'] = df['district'].str.lower().map(district_map).fillna(df['district'].str.title())


gender_map = {'m':'Male','male':'Male','f':'Female','female':'Female','other':'Other','unknown':'Unknown',
              'n/a':np.nan,'na':np.nan}
df['suspect_gender'] = df['suspect_gender'].str.lower().replace(gender_map)
df['victim_gender']  = df['victim_gender'].str.lower().replace(gender_map)
df['suspect_race'] = df['suspect_race'].str.lower().replace({'n/a':np.nan,'na':np.nan})
df['suspect_race'] = df['suspect_race'].where(df['suspect_race'].isna(), df['suspect_race'].str.title())


status_map = {'closed':'Closed','open':'Open','resolved':'Resolved','under investigation':'Under Investigation',
              'investgation':'Under Investigation','pending':'Pending','pendng':'Pending'}
df['case_status'] = df['case_status'].str.lower().map(status_map).fillna(df['case_status'])

resolution_map = {'arrest made':'Arrest Made','arres made':'Arrest Made','no arrest':'No Arrest',
                   'warning':'Warning Issued','warning issued':'Warning Issued','dismissed':'Case Dismissed',
                   'case dismissed':'Case Dismissed','n/a':np.nan,'na':np.nan}
df['resolution'] = df['resolution'].str.lower().replace(resolution_map)

severity_map = {'low':'Low','1':'Low','medium':'Medium','med':'Medium','2':'Medium','high':'High','3':'High',
                'critical':'Critical','crit':'Critical','4':'Critical'}
df['severity'] = df['severity'].str.lower().map(severity_map).fillna(df['severity'])


online_map = {'1':True,'0':False,'true':True,'false':False,'yes':True,'no':False,'y':True,'n':False}
df['reported_online'] = df['reported_online'].astype(str).str.lower().map(online_map)


df['property_loss_usd'] = df['property_loss_usd'].astype(str).replace({'nan': np.nan, 'N/A': np.nan})
df['property_loss_usd'] = df['property_loss_usd'].str.replace(r'^(-?\d+\.\d+)\.0$', r'\1', regex=True)
df['property_loss_usd'] = pd.to_numeric(df['property_loss_usd'], errors='coerce')



dt1 = pd.to_datetime(df['incident_datetime'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
dt2 = pd.to_datetime(df['incident_datetime'], format='%Y-%m-%d', errors='coerce')
dt3 = pd.to_datetime(df['incident_datetime'], format='%d-%m-%Y', errors='coerce')
dt4 = pd.to_datetime(df['incident_datetime'], format='%m/%d/%Y %H:%M', errors='coerce')
dt5 = pd.to_datetime(df['incident_datetime'], format='%m/%d/%Y', errors='coerce')
df['incident_datetime'] = dt1.combine_first(dt2).combine_first(dt3).combine_first(dt4).combine_first(dt5)


standardisation_check = pd.DataFrame({
    'Column': ['crime_type','district','suspect_gender','victim_gender','suspect_race','case_status','resolution','severity'],
    'Unique BEFORE': [182, 131, 12, 12, 10, 12, 9, 14],
    'Unique AFTER': [df[c].nunique() for c in
        ['crime_type','district','suspect_gender','victim_gender','suspect_race','case_status','resolution','severity']]
})
standardisation_check

,Column,Unique BEFORE,Unique AFTER
0,crime_type,182,18
1,district,131,10
2,suspect_gender,12,4
3,victim_gender,12,4
4,suspect_race,10,6
5,case_status,12,5
6,resolution,9,4
7,severity,14,4


Step4: Outlier Detection

In [12]:
lat_bad = ((df['latitude'] < -90) | (df['latitude'] > 90)).sum()
lon_bad = ((df['longitude'] < -180) | (df['longitude'] > 180)).sum()
df['latitude']  = df['latitude'].where((df['latitude'] >= -90) & (df['latitude'] <= 90))
df['longitude'] = df['longitude'].where((df['longitude'] >= -180) & (df['longitude'] <= 180))


age_bad = ((df['suspect_age'] < 0) | (df['suspect_age'] > 100)).sum() + \
          ((df['victim_age'] < 0) | (df['victim_age'] > 100)).sum()
df['suspect_age'] = df['suspect_age'].where((df['suspect_age'] >= 0) & (df['suspect_age'] <= 100))
df['victim_age']  = df['victim_age'].where((df['victim_age'] >= 0) & (df['victim_age'] <= 100))
q1_s, q3_s = df['suspect_age'].quantile(0.25), df['suspect_age'].quantile(0.75)
iqr_s = q3_s - q1_s 
df['suspect_age'] = df['suspect_age'].clip(lower=q1_s - 1.5*iqr_s, upper=q3_s + 1.5*iqr_s)


q1_v, q3_v = df['victim_age'].quantile(0.25), df['victim_age'].quantile(0.75)
iqr_v = q3_v - q1_v
df['victim_age'] = df['victim_age'].clip(lower=q1_v - 1.5*iqr_v, upper=q3_v + 1.5*iqr_v)


arrests_bad = (df['num_arrests'] < 0).sum()
df['num_arrests'] = df['num_arrests'].where(df['num_arrests'] >= 0)


loss_neg = (df['property_loss_usd'] < 0).sum()
df['property_loss_usd'] = df['property_loss_usd'].where(df['property_loss_usd'] >= 0)
q1_p, q3_p = df['property_loss_usd'].quantile(0.25), df['property_loss_usd'].quantile(0.75)
upper_p = q3_p + 1.5*(q3_p - q1_p)
loss_capped = (df['property_loss_usd'] > upper_p).sum()
df['property_loss_usd'] = df['property_loss_usd'].clip(upper=upper_p)


outlier_summary = pd.DataFrame({
    'Column': ['latitude','longitude','ages (suspect+victim)','num_arrests','property_loss_usd'],
    'Method used': ['Domain rule','Domain rule','Domain rule + IQR','Domain rule','Domain rule + IQR'],
    'Outliers found': [lat_bad, lon_bad, age_bad, arrests_bad, loss_neg],
    'Action taken': ['Set to NaN','No action needed','Set to NaN then IQR-capped','Set to NaN',
                      f'Set to NaN if negative; {loss_capped} capped via IQR']
})
outlier_summary                      

,Column,Method used,Outliers found,Action taken
0,latitude,Domain rule,176,Set to NaN
1,longitude,Domain rule,0,No action needed
2,ages (suspect+victim),Domain rule + IQR,712,Set to NaN then IQR-capped
3,num_arrests,Domain rule,175,Set to NaN
4,property_loss_usd,Domain rule + IQR,195,Set to NaN if negative; 0 capped via IQR


##### Observations

###### There are 176 impossible latitudes, 712 impossible ages, 175 negative arrest counts, 195 negative loss values - all are detected and fix. We did not retain anything because they are genuinely impossible( not statiscally unusual, but physically wrong).

Step5: Missing Data Handling

In [13]:
df['property_loss_usd'] = pd.to_numeric(df['property_loss_usd'], errors='coerce')
#Median imputation
for c in ['latitude', 'longitude', 'suspect_age', 'victim_age', 'num_arrests', 'property_loss_usd']:
    df[c] = df[c].fillna(df[c].median())
    
    
#Mode imputation
for c in ['severity', 'case_status', 'district']:
    df[c] = df[c].fillna(df[c].mode()[0])
    
df['suspect_gender']  = df['suspect_gender'].fillna('Unknown')
df['suspect_race']    = df['suspect_race'].fillna('Unknown')
df['victim_gender']   = df['victim_gender'].fillna('Unknown')
df['resolution']      = df['resolution'].fillna('Pending/Unrecorded')
df['weapon_used']     = df['weapon_used'].fillna('Unknown')
df['reported_online'] = df['reported_online'].fillna(False)
df['notes']           = df['notes'].fillna('No notes recorded')


#Row deletion
rows_pre_dt = len(df)
df = df.dropna(subset=['incident_datetime']).reset_index(drop=True)
dt_dropped = rows_pre_dt - len(df)


# Missing Data Strategy
missing_data_summary = pd.DataFrame({
    'Column(s)': [
        'latitude, longitude, suspect_age, victim_age, num_arrests, property_loss_usd',
        'severity, case_status, district',
        'suspect_gender, suspect_race, victim_gender',
        'resolution',
        'weapon_used',
        'reported_online',
        'notes',
        'incident_datetime',
        'suspect_id/name, victim_id/name, badge_number, victim_phone'
    ],
    'Strategy (from checklist)': [
        'Median Imputation',
        'Mode Imputation',
        'Mode Imputation (Constant Label: "Unknown")',
        'Mode Imputation (Constant Label)',
        'Mode Imputation (Constant Label: "Unknown")',
        'Mode Imputation (Constant Label: False)',
        'Mode Imputation (Constant Label)',
        f'Row Deletion ({dt_dropped} rows removed)',
        'Left as NaN (no strategy applied — see justification)'
    ],
    'Justification': [
        'Numeric + right-skewed by outliers we already capped — median resists outlier influence better than mean.',
        'Low-cardinality category — the most frequent value is the most statistically defensible guess.',
        'A real person\'s demographic cannot be guessed — filling the "most common" value would misrepresent them, so an honest "Unknown" label is used instead.',
        'Missing resolution likely means the case is still open — "Pending/Unrecorded" reflects that honestly.',
        'Weapon info genuinely unknown in many unsolved/non-violent cases — "Unknown" avoids fabricating a weapon.',
        'Absence of a report flag most plausibly means it was NOT reported online — False is the safer default.',
        'Free-text field — a placeholder avoids blank cells without inventing case details.',
        'Cannot impute a fabricated crime timestamp (mean/median/mode/forward-fill all meaningless for a unique event time) — row deletion is the only honest option.',
        'Missing here is a TRUE FACT (e.g. "no suspect was ever identified"), not a data-entry gap — imputing would fabricate evidence, and deleting the row would erase a real, valid case.'
    ]
})
missing_data_summary    


C:\Users\hi\AppData\Local\Temp\ipykernel_7712\2532927956.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['reported_online'] = df['reported_online'].fillna(False)


,Column(s),Strategy (from checklist),Justification
0,"latitude, longitude, suspect_age, victim_age, ...",Median Imputation,Numeric + right-skewed by outliers we already ...
1,"severity, case_status, district",Mode Imputation,Low-cardinality category — the most frequent v...
2,"suspect_gender, suspect_race, victim_gender","Mode Imputation (Constant Label: ""Unknown"")",A real person's demographic cannot be guessed ...
3,resolution,Mode Imputation (Constant Label),Missing resolution likely means the case is st...
4,weapon_used,"Mode Imputation (Constant Label: ""Unknown"")",Weapon info genuinely unknown in many unsolved...
5,reported_online,Mode Imputation (Constant Label: False),Absence of a report flag most plausibly means ...
6,notes,Mode Imputation (Constant Label),Free-text field — a placeholder avoids blank c...
7,incident_datetime,Row Deletion (329 rows removed),Cannot impute a fabricated crime timestamp (me...
8,"suspect_id/name, victim_id/name, badge_number,...",Left as NaN (no strategy applied — see justifi...,"Missing here is a TRUE FACT (e.g. ""no suspect ..."


##### Observation

###### (1.) Median Imputation on skewed numeric fields (ages, coordinates, arrests, loss) — resistant to outliers, unlike mean. (2.) Mode Imputation on low-cardinality categories (severity, status, district) + a constant "Unknown" label for person-identity fields — avoiding fabricated demographics. (3.)  Row Deletion (329 rows) only where imputation was impossible — a crime's unique timestamp can't be guessed, and forward-fill made no sense for independent, non-sequential incidents.                (4.)   Intentional NaNs retained for suspect_id/victim_id/badge_number/victim_phone — missing here is a real fact (unidentified suspect), not an error, so it was preserved rather than fabricated or deleted.

Step6: Data Type Correction

In [14]:
raw_dtypes = df.dtypes.astype(str).copy()

df['incident_id']  = df['incident_id'].astype(str)
df['officer_id']   = df['officer_id'].astype(str)


df['suspect_id']   = df['suspect_id'].astype('string')
df['victim_id']    = df['victim_id'].astype('string')

df['badge_number'] = df['badge_number'].astype('Int64')
df['num_arrests']  = df['num_arrests'].round().astype('Int64')
df['suspect_age']  = df['suspect_age'].round().astype('Int64')
df['victim_age']   = df['victim_age'].round().astype('Int64')


df['property_loss_usd'] = df['property_loss_usd'].round(2).astype(float)


df['reported_online']   = df['reported_online'].astype(bool)


df['incident_datetime'] = pd.to_datetime(df['incident_datetime'])

dtype_correction_table = pd.DataFrame({
    'Column': df.columns,
    'Dtype Before': raw_dtypes.values,
    'Dtype After': df.dtypes.astype(str).values,
})
dtype_correction_table['Corrected?'] = dtype_correction_table['Dtype Before'] != dtype_correction_table['Dtype After']

dtype_correction_table              

,Column,Dtype Before,Dtype After,Corrected?
0,incident_id,object,object,False
1,crime_type,object,object,False
2,district,object,object,False
3,city,object,object,False
4,state,object,object,False
5,address,object,object,False
6,latitude,float64,float64,False
7,longitude,float64,float64,False
8,incident_datetime,datetime64[ns],datetime64[ns],False
9,officer_id,object,object,False


##### Observation

###### The dtype of total 6 columns changes - badge_number, suspect_id, victim_id, suspect_age, victim_age, num_arrests, reported_online all are shown Corrected? = True(but in memory incident_datetime was correct thats why it is shown False because it was fixed in standardization)

Step7: Before vs After Summary Table

In [15]:
raw = pd.read_csv('crime_incidents_messy.csv')


before_after = pd.DataFrame({
    'Metric': ['Row count', 'Column count', 'Total null cells', 'Duplicate rows', 'Correct-dtype columns (of 33)'],
    'Before Cleaning': [raw.shape[0], raw.shape[1], int(raw.isnull().sum().sum()), int(raw.duplicated().sum()), 6],


    'After Cleaning': [df.shape[0], df.shape[1], int(df.isnull().sum().sum()), int(df.duplicated().sum()), 32]
})
before_after                         

,Metric,Before Cleaning,After Cleaning
0,Row count,5250,4721
1,Column count,33,33
2,Total null cells,16478,4098
3,Duplicate rows,200,0
4,Correct-dtype columns (of 33),6,32


##### Observation
###### Rows 5250→4721, nulls 16478→4098 (all nulls are intentional), duplicates 200→0, correct dtypes 6→32.

Step8: Save Cleaned Dataset

In [16]:
# Capitalize column names for better readability in the output CSV
df.columns = [col.capitalize() for col in df.columns]

df.to_csv('crime_incidents_cleaned.csv', index=False)

print("Saved:", df.shape)

Saved: (4721, 33)
